# 04. Cálculo de Reservas Técnicas al 31 de diciembre de 2025

En este cuaderno se estiman y comparan las reservas técnicas para el **mercado total** de seguros marítimos con fecha de valuación al **31 de diciembre de 2025**, expresadas en pesos constantes de diciembre de 2025:

1. **Inciso 5 — Reserva de Riesgos en Curso (RRC):**
   - **Método exacto** (póliza por póliza según días exactos por devengar sobre la prima de tarifa).
   - **Método de los veinticuatroavos** (aproximación mensual con factores $(2m-1)/24$ sobre la prima de tarifa).
   - **Método estatutario** (mejor estimación actuarial basada en la Prima No Devengada multiplicada por el factor de siniestralidad histórica media del mercado, $\text{BEL} = \text{PND} \times \bar{S}$).
   - **Cuadro comparativo de la RRC** con diferencias absolutas y porcentuales.

2. **Inciso 6 — Reserva de Obligaciones Pendientes de Cumplir por Siniestros Ocurridos y No Reportados (SONR / IBNR):**
   - **Método Chain-Ladder para el mercado total** (triángulo agregado sin conglomerados, con factores ponderados por volumen y promedio simple; dejando pendiente la desagregación por conglomerados).
   - **Método de la razón** (basado en la razón de siniestralidad esperada respecto a la prima emitida, tanto en su formulación ponderada por desarrollo pendiente / Bornhuetter-Ferguson como en diferencia directa $\text{Prima} \times \bar{S} - \text{Pagado}$).
   - **Método estatutario** (aplicación de factores estatutarios de desarrollo no reportado sobre las primas emitidas y siniestros del ejercicio).
   - **Cuadro comparativo de la SONR** con diferencias absolutas y porcentuales.

3. **Inciso 7 — Conclusiones actuariales:**
   - Interpretación técnica de las diferencias entre metodologías e integración con los hallazgos de los apartados anteriores.

In [1]:
# ------------------------------------------
# 1. Importación de librerías y configuración
# ------------------------------------------
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "funciones_reservas.py").exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto con funciones_reservas.py.")

sys.path.insert(0, str(ROOT))

from funciones_reservas import (
    calcular_rrc_exacto,
    calcular_rrc_veinticuatroavos,
    construir_triangulos_desarrollo,
    calcular_sonr_chain_ladder,
    normalizar_categorias,
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
fecha_valoracion = pd.Timestamp("2025-12-31")

# ------------------------------------------
# 2. Carga de bases de datos
# ------------------------------------------
archivo_2025 = RAW / "EMISION 2025.xlsx"
emision_2025 = pd.read_excel(archivo_2025, sheet_name=0)
emision_ajustada = pd.read_parquet(PROCESSED / "emision_ajustada_2025.parquet")
siniestros_ajustados = pd.read_parquet(PROCESSED / "siniestros_ajustados_2025.parquet")

# Normalizar categorías y fechas en EMISION 2025
emision_2025[["tipo_embarcacion", "uso"]] = normalizar_categorias(
    emision_2025, ["tipo_embarcacion", "uso"]
)[["tipo_embarcacion", "uso"]]

for columna in ["inicio_vigencia", "fin_vigencia"]:
    emision_2025[columna] = pd.to_datetime(emision_2025[columna], errors="raise")

# Definir la Prima de Tarifa (Prima Emitida neta de Comisión Directa)
emision_2025["prima_tarifa"] = emision_2025["prima_emitida"] - emision_2025["comision_directa"]

print(f"Fecha de valuación: {fecha_valoracion.date()}")
print(f"Dimensiones EMISION 2025:      {emision_2025.shape}")
print(f"Dimensiones Emisión Ajustada:  {emision_ajustada.shape}")
print(f"Dimensiones Siniestros Ajust.: {siniestros_ajustados.shape}")
display(emision_2025.head())

Fecha de valuación: 2025-12-31
Dimensiones EMISION 2025:      (1982, 10)
Dimensiones Emisión Ajustada:  (21353, 11)
Dimensiones Siniestros Ajust.: (2763, 14)


,año,tipo_embarcacion,uso,anio_construccion,prima_emitida,prima_retenida,comision_directa,inicio_vigencia,fin_vigencia,prima_tarifa
0,2025,remolcador,remolcador,2000,-818409,0,0,2025-07-15,2026-05-29,-818409
1,2025,carguero general,comercial,1995,-297013,0,0,2025-03-25,2025-12-20,-297013
2,2025,abastecedor,comercial,2017,-8094,0,0,2025-12-21,2026-04-14,-8094
3,2025,abastecedor,comercial,2018,-6969,0,0,2025-11-23,2026-03-09,-6969
4,2025,abastecedor,comercial,2021,-6380,0,0,2025-01-17,2025-04-21,-6380


### Verificación de la cartera de emisión 2025

Antes de calcular la Reserva de Riesgos en Curso (RRC), verificamos la consistencia de `EMISION 2025.xlsx`:
- Todos los registros corresponden al año de emisión **2025**, por lo que sus montos ya se encuentran expresados en pesos de diciembre de 2025 ($\text{factor INPC} = 1.0$).
- La **prima de tarifa** (prima de riesgo más gastos de administración, descontando costos de adquisición) se obtiene como:
  $$\text{Prima de Tarifa}_k = \text{Prima Emitida}_k - \text{Comisión Directa}_k$$
- Se conservan los registros con primas negativas (cancelaciones, endosos o devoluciones de prima) para reflejar el compromiso neto real del mercado al cierre del ejercicio.

In [2]:
# ------------------------------------------
# Resumen de vigencias e importes en 2025
# ------------------------------------------
duracion_dias = (emision_2025["fin_vigencia"] - emision_2025["inicio_vigencia"]).dt.days
vigentes_cierre = emision_2025["fin_vigencia"] > fecha_valoracion

resumen_cartera_2025 = pd.DataFrame({
    "Indicador": [
        "Total de registros en 2025",
        "Pólizas con vigencia anual (365 días)",
        "Registros con duración < 365 días (ajustes/endosos)",
        "Pólizas vigentes después del 31/12/2025",
        "Pólizas cuya vigencia concluyó antes o el 31/12/2025",
        "Prima Emitida Total 2025 (MXN)",
        "Comisión Directa Total 2025 (MXN)",
        "Prima de Tarifa Total 2025 (Prima Emitida - Comisión) (MXN)",
        "Prima Retenida Total 2025 (MXN)",
    ],
    "Valor": [
        f"{len(emision_2025):,}",
        f"{(duracion_dias == 365).sum():,}",
        f"{(duracion_dias < 365).sum():,}",
        f"{vigentes_cierre.sum():,}",
        f"{(~vigentes_cierre).sum():,}",
        f"${emision_2025['prima_emitida'].sum():,.2f}",
        f"${emision_2025['comision_directa'].sum():,.2f}",
        f"${emision_2025['prima_tarifa'].sum():,.2f}",
        f"${emision_2025['prima_retenida'].sum():,.2f}",
    ]
})
display(resumen_cartera_2025)

,Indicador,Valor
0,Total de registros en 2025,"1,982"
1,Pólizas con vigencia anual (365 días),"1,895"
2,Registros con duración < 365 días (ajustes/end...,87
3,Pólizas vigentes después del 31/12/2025,"1,939"
4,Pólizas cuya vigencia concluyó antes o el 31/1...,43
5,Prima Emitida Total 2025 (MXN),"$6,151,443,214.00"
6,Comisión Directa Total 2025 (MXN),"$72,758,282.00"
7,Prima de Tarifa Total 2025 (Prima Emitida - Co...,"$6,078,684,932.00"
8,Prima Retenida Total 2025 (MXN),"$230,886,331.00"


---
# 5. Reserva de Riesgos en Curso (RRC) al 31 de diciembre de 2025

La **Reserva de Riesgos en Curso (RRC)** tiene como propósito cubrir el valor esperado de las obligaciones futuras derivadas de las pólizas que permanecen en vigor después de la fecha de valuación ($t_{\text{val}} = \text{31 de diciembre de 2025}$).

## 5.1. Método Exacto (Póliza por Póliza)

El **método exacto** calcula para cada póliza $k$ la proporción exacta de días de cobertura aún no devengados con posterioridad al 31 de diciembre de 2025 respecto a la duración total de la vigencia de la póliza:

$$p_k^{\text{exacto}} = \frac{\max\left(0,\; \text{Fin Vigencia}_k - t_{\text{val}}\right)}{\text{Fin Vigencia}_k - \text{Inicio Vigencia}_k}, \qquad \text{RRC}_k^{\text{Exacto}} = \text{Prima de Tarifa}_k \times p_k^{\text{exacto}}$$

donde $\text{Prima de Tarifa}_k = \text{Prima Emitida}_k - \text{Comisión Directa}_k$. Adicionalmente, se reporta como referencia el resultado sobre la prima emitida directa sin descontar comisiones.

In [3]:
# ------------------------------------------
# 5.1 Cálculo de la RRC por el Método Exacto
# ------------------------------------------
df_rrc_exacto = calcular_rrc_exacto(
    emision_2025,
    col_prima="prima_tarifa",
    fecha_valuacion=fecha_valoracion,
)

# Cálculo de referencia sobre prima_emitida directa
df_rrc_exacto["rrc_exacto_emitida"] = (
    df_rrc_exacto["prima_emitida"] * df_rrc_exacto["proporcion_no_devengada_exacta"]
)

rrc_exacto_tarifa = df_rrc_exacto["rrc_exacto"].sum()
rrc_exacto_emitida = df_rrc_exacto["rrc_exacto_emitida"].sum()

print(f"RRC Método Exacto (sobre Prima de Tarifa):  ${rrc_exacto_tarifa:,.2f}")
print(f"RRC Método Exacto (sobre Prima Emitida):    ${rrc_exacto_emitida:,.2f}")
print(f"Proporción media ponderada no devengada:    {rrc_exacto_tarifa / emision_2025['prima_tarifa'].sum():.4%}")

# Muestra de las primeras pólizas con su proporción exacta
columnas_muestra_exacto = [
    "tipo_embarcacion", "inicio_vigencia", "fin_vigencia",
    "duracion_dias", "dias_por_devengar", "proporcion_no_devengada_exacta",
    "prima_tarifa", "rrc_exacto"
]
display(df_rrc_exacto[columnas_muestra_exacto].head(10).round({"proporcion_no_devengada_exacta": 4, "rrc_exacto": 2}))

RRC Método Exacto (sobre Prima de Tarifa):  $5,244,839,068.74
RRC Método Exacto (sobre Prima Emitida):    $5,282,060,815.21
Proporción media ponderada no devengada:    86.2825%


,tipo_embarcacion,inicio_vigencia,fin_vigencia,duracion_dias,dias_por_devengar,proporcion_no_devengada_exacta,prima_tarifa,rrc_exacto
0,remolcador,2025-07-15,2026-05-29,318,149,0.4686,-818409,-383468.3679
1,carguero general,2025-03-25,2025-12-20,270,0,0.0000,-297013,-0.0000
2,abastecedor,2025-12-21,2026-04-14,114,104,0.9123,-8094,-7384.0000
3,abastecedor,2025-11-23,2026-03-09,106,68,0.6415,-6969,-4470.6792
4,abastecedor,2025-01-17,2025-04-21,94,0,0.0000,-6380,-0.0000
5,abastecedor,2025-02-27,2025-05-20,82,0,0.0000,-4876,-0.0000
6,abastecedor,2025-08-14,2025-10-07,54,0,0.0000,-2837,-0.0000
7,abastecedor,2025-03-01,2025-04-04,34,0,0.0000,-1354,-0.0000
8,abastecedor,2025-12-05,2025-12-27,22,0,0.0000,-1190,-0.0000
9,carguero general,2025-08-18,2026-08-18,365,230,0.6301,4,2.5205


## 5.2. Método de los Veinticuatroavos

El **método de los veinticuatroavos** es una aproximación tradicional que supone que todas las pólizas emitidas en el mes $m \in \{1, 2, \dots, 12\}$ son anuales y se suscriben uniformemente a lo largo del mes (equivalentemente, a mitad del mes $m$, es decir, en el quinceno $2m-1$ de los 24 medios meses del año).

Al 31 de diciembre del año de emisión, la fracción **devengada** de las pólizas emitidas en el mes $m$ es $\frac{25 - 2m}{24}$, mientras que la fracción **no devengada** (que constituye la RRC) es su complemento:

$$f_m^{24\text{avos}} = 1 - \frac{25 - 2m}{24} = \frac{2m - 1}{24}, \qquad m = 1, 2, \dots, 12$$

Por lo tanto, la RRC bajo el método de los veinticuatroavos se obtiene como:

$$\text{RRC}_{24\text{avos}} = \sum_{m=1}^{12} \text{Prima de Tarifa}_m \times \frac{2m - 1}{24}$$

In [4]:
# ------------------------------------------
# 5.2 Cálculo de la RRC por Veinticuatroavos
# ------------------------------------------
df_rrc = calcular_rrc_veinticuatroavos(
    df_rrc_exacto,
    col_prima="prima_tarifa",
)
df_rrc["rrc_24avos_emitida"] = (
    df_rrc["prima_emitida"] * df_rrc["factor_no_devengado_24avos"]
)

rrc_24avos_tarifa = df_rrc["rrc_24avos"].sum()
rrc_24avos_emitida = df_rrc["rrc_24avos_emitida"].sum()

# Tabla resumen por mes de emisión
nombres_meses = {
    1: "01 - Enero", 2: "02 - Febrero", 3: "03 - Marzo", 4: "04 - Abril",
    5: "05 - Mayo", 6: "06 - Junio", 7: "07 - Julio", 8: "08 - Agosto",
    9: "09 - Septiembre", 10: "10 - Octubre", 11: "11 - Noviembre", 12: "12 - Diciembre"
}

tabla_mensual_rrc = (
    df_rrc.groupby("mes_emision", as_index=False)
    .agg(
        polizas=("prima_tarifa", "count"),
        prima_emitida=("prima_emitida", "sum"),
        comision_directa=("comision_directa", "sum"),
        prima_tarifa=("prima_tarifa", "sum"),
        factor_24avos=("factor_no_devengado_24avos", "first"),
        rrc_24avos=("rrc_24avos", "sum"),
        rrc_exacto=("rrc_exacto", "sum"),
    )
)
tabla_mensual_rrc["Mes"] = tabla_mensual_rrc["mes_emision"].map(nombres_meses)
tabla_mensual_rrc["Fracción"] = tabla_mensual_rrc["mes_emision"].map(lambda m: f"{2*m - 1}/24")
tabla_mensual_rrc["Dif_24avos_vs_Exacto"] = tabla_mensual_rrc["rrc_24avos"] - tabla_mensual_rrc["rrc_exacto"]

print(f"RRC Método de Veinticuatroavos (sobre Prima de Tarifa): ${rrc_24avos_tarifa:,.2f}")
print(f"RRC Método de Veinticuatroavos (sobre Prima Emitida):   ${rrc_24avos_emitida:,.2f}")

columnas_mostrar_mes = [
    "Mes", "polizas", "Fracción", "factor_24avos",
    "prima_tarifa", "rrc_exacto", "rrc_24avos", "Dif_24avos_vs_Exacto"
]
display(
    tabla_mensual_rrc[columnas_mostrar_mes].round({
        "factor_24avos": 4,
        "prima_tarifa": 2,
        "rrc_exacto": 2,
        "rrc_24avos": 2,
        "Dif_24avos_vs_Exacto": 2,
    })
)

RRC Método de Veinticuatroavos (sobre Prima de Tarifa): $5,284,812,484.50
RRC Método de Veinticuatroavos (sobre Prima Emitida):   $5,322,232,615.25


,Mes,polizas,Fracción,factor_24avos,prima_tarifa,rrc_exacto,rrc_24avos,Dif_24avos_vs_Exacto
0,01 - Enero,140,1/24,0.0417,94220347,2.909563e+06,3.925848e+06,1016284.52
1,02 - Febrero,160,3/24,0.1250,88094029,1.217252e+07,1.101175e+07,-1160769.93
2,03 - Marzo,197,5/24,0.2083,124201082,2.537818e+07,2.587523e+07,497041.48
3,04 - Abril,148,7/24,0.2917,24832773,7.739757e+06,7.242892e+06,-496864.48
4,05 - Mayo,150,9/24,0.3750,104838272,3.853588e+07,3.931435e+07,778470.78
5,06 - Junio,164,11/24,0.4583,138489011,6.418490e+07,6.347413e+07,-710772.85
6,07 - Julio,162,13/24,0.5417,139487519,7.500630e+07,7.555574e+07,549440.17
7,08 - Agosto,164,15/24,0.6250,80029744,5.000307e+07,5.001859e+07,15524.10
8,09 - Septiembre,188,17/24,0.7083,85486189,6.035027e+07,6.055272e+07,202446.20
9,10 - Octubre,169,19/24,0.7917,129633705,1.013811e+08,1.026267e+08,1245631.88


## 5.3. Método Estatutario para la RRC

Bajo el enfoque estatutario de mejor estimación actuarial (*Best Estimate Liability*, $\text{BEL}$), la reserva técnica de riesgos en curso busca reflejar el monto esperado de siniestralidad futura asociada a la porción no devengada de las primas en vigor:

$$\text{RRC}_{\text{Estatutario}} = \text{PND} \times \bar{S}$$

donde:
- $\text{PND}$ es la **Prima No Devengada** al 31 de diciembre de 2025 (valuada sobre la prima de tarifa mediante el método exacto, y reportando también su variante con veinticuatroavos y sobre prima emitida).
- $\bar{S}$ es el **factor de siniestralidad histórica promedio del mercado**, calculado a partir de las bases ajustadas por inflación a diciembre de 2025:
  - Considerando los **ocho años de origen con desarrollo completo (2015–2022)**: $\bar{S}_{2015\text{–}2022} = 0.650849$ ($65.08\%$).
  - Considerando **todos los años de la serie histórica (2015–2025)**: $\bar{S}_{2015\text{–}2025} = 0.611773$ ($61.18\%$).

In [5]:
# ------------------------------------------
# 5.3 Cálculo de la RRC por el Método Estatutario
# ------------------------------------------
columnas_pagos_ajustados = [f"{i}_monto pagado_ajustada" for i in range(4)]

siniestros_anual = (
    siniestros_ajustados[columnas_pagos_ajustados]
    .sum(axis=1, min_count=1)
    .groupby(siniestros_ajustados["año"])
    .sum(min_count=1)
)
primas_anual = (
    emision_ajustada.groupby("año")["prima_emitida_ajustada"]
    .sum(min_count=1)
)
factores_siniestralidad = siniestros_anual / primas_anual

# Promedios históricos de siniestralidad
s_media_2015_2022 = float(factores_siniestralidad.loc[2015:2022].mean())
s_media_2015_2025 = float(factores_siniestralidad.loc[2015:2025].mean())
s_pond_2015_2022 = float(siniestros_anual.loc[2015:2022].sum() / primas_anual.loc[2015:2022].sum())

# Cálculo principal: PND (Exacto, Prima de Tarifa) * Siniestralidad media histórica (2015-2022 y 2015-2025)
rrc_estatutario_2015_2022 = rrc_exacto_tarifa * s_media_2015_2022
rrc_estatutario_2015_2025 = rrc_exacto_tarifa * s_media_2015_2025

# Variantes de referencia sobre Prima Emitida y con PND 24avos
rrc_estatutario_emitida_15_22 = rrc_exacto_emitida * s_media_2015_2022
rrc_estatutario_emitida_15_25 = rrc_exacto_emitida * s_media_2015_2025
rrc_estatutario_24avos_15_22 = rrc_24avos_tarifa * s_media_2015_2022

print(f"Factor de siniestralidad media (2015–2022, maduros): {s_media_2015_2022:.6f} ({s_media_2015_2022:.2%})")
print(f"Factor de siniestralidad media (2015–2025, total):   {s_media_2015_2025:.6f} ({s_media_2015_2025:.2%})")
print("-" * 75)
print(f"RRC Estatutario (PND Exacto Tarifa × S_media 2015–2022):  ${rrc_estatutario_2015_2022:,.2f}")
print(f"RRC Estatutario (PND Exacto Tarifa × S_media 2015–2025):  ${rrc_estatutario_2015_2025:,.2f}")
print(f"RRC Estatutario (PND 24avos Tarifa × S_media 2015–2022):  ${rrc_estatutario_24avos_15_22:,.2f}")
print(f"RRC Estatutario (PND Exacto Emitida × S_media 2015–2022): ${rrc_estatutario_emitida_15_22:,.2f}")

Factor de siniestralidad media (2015–2022, maduros): 0.650849 (65.08%)
Factor de siniestralidad media (2015–2025, total):   0.611773 (61.18%)
---------------------------------------------------------------------------
RRC Estatutario (PND Exacto Tarifa × S_media 2015–2022):  $3,413,597,990.56
RRC Estatutario (PND Exacto Tarifa × S_media 2015–2025):  $3,208,653,187.09
RRC Estatutario (PND 24avos Tarifa × S_media 2015–2022):  $3,439,614,646.16
RRC Estatutario (PND Exacto Emitida × S_media 2015–2022): $3,437,823,725.09


## 5.4. Cuadro Comparativo de la Reserva de Riesgos en Curso (RRC)

A continuación se presenta el cuadro comparativo de los tres métodos solicitados para la RRC al 31 de diciembre de 2025, tomando como base principal la **Prima de Tarifa** ($\text{Prima Emitida} - \text{Comisión Directa} = \$6,078,684,932.00$) e incluyendo también la comparación sobre la **Prima Emitida Directa** ($\$6,151,443,214.00$).

In [6]:
# ------------------------------------------
# 5.4 Cuadro comparativo de RRC al 31/12/2025
# ------------------------------------------
prima_tarifa_total = float(emision_2025["prima_tarifa"].sum())
prima_emitida_total = float(emision_2025["prima_emitida"].sum())

cuadro_rrc = pd.DataFrame([
    {
        "Método": "1. Método Exacto (Póliza por póliza)",
        "Criterio / Fórmula": "Σ Prima_Tarifa_k × (Días por devengar_k / Duración_k)",
        "RRC sobre Prima de Tarifa (MXN)": rrc_exacto_tarifa,
        "% de Prima de Tarifa": (rrc_exacto_tarifa / prima_tarifa_total) * 100,
        "Dif. vs Exacto (MXN)": 0.0,
        "Dif. vs Exacto (%)": 0.0,
        "RRC sobre Prima Emitida (MXN)": rrc_exacto_emitida,
    },
    {
        "Método": "2. Método de los Veinticuatroavos",
        "Criterio / Fórmula": "Σ Prima_Tarifa_m × (2m - 1) / 24",
        "RRC sobre Prima de Tarifa (MXN)": rrc_24avos_tarifa,
        "% de Prima de Tarifa": (rrc_24avos_tarifa / prima_tarifa_total) * 100,
        "Dif. vs Exacto (MXN)": rrc_24avos_tarifa - rrc_exacto_tarifa,
        "Dif. vs Exacto (%)": ((rrc_24avos_tarifa / rrc_exacto_tarifa) - 1) * 100,
        "RRC sobre Prima Emitida (MXN)": rrc_24avos_emitida,
    },
    {
        "Método": "3a. Método Estatutario (S̄ 2015–2022 = 65.08%)",
        "Criterio / Fórmula": "PND_Exacto × S̄_2015-2022 (años con desarrollo completo)",
        "RRC sobre Prima de Tarifa (MXN)": rrc_estatutario_2015_2022,
        "% de Prima de Tarifa": (rrc_estatutario_2015_2022 / prima_tarifa_total) * 100,
        "Dif. vs Exacto (MXN)": rrc_estatutario_2015_2022 - rrc_exacto_tarifa,
        "Dif. vs Exacto (%)": ((rrc_estatutario_2015_2022 / rrc_exacto_tarifa) - 1) * 100,
        "RRC sobre Prima Emitida (MXN)": rrc_estatutario_emitida_15_22,
    },
    {
        "Método": "3b. Método Estatutario (S̄ 2015–2025 = 61.18%)",
        "Criterio / Fórmula": "PND_Exacto × S̄_2015-2025 (todos los años observados)",
        "RRC sobre Prima de Tarifa (MXN)": rrc_estatutario_2015_2025,
        "% de Prima de Tarifa": (rrc_estatutario_2015_2025 / prima_tarifa_total) * 100,
        "Dif. vs Exacto (MXN)": rrc_estatutario_2015_2025 - rrc_exacto_tarifa,
        "Dif. vs Exacto (%)": ((rrc_estatutario_2015_2025 / rrc_exacto_tarifa) - 1) * 100,
        "RRC sobre Prima Emitida (MXN)": rrc_estatutario_emitida_15_25,
    },
])

cuadro_rrc_fmt = cuadro_rrc.copy()
cuadro_rrc_fmt["RRC sobre Prima de Tarifa (MXN)"] = cuadro_rrc_fmt["RRC sobre Prima de Tarifa (MXN)"].map("${:,.2f}".format)
cuadro_rrc_fmt["% de Prima de Tarifa"] = cuadro_rrc_fmt["% de Prima de Tarifa"].map("{:.2f}%".format)
cuadro_rrc_fmt["Dif. vs Exacto (MXN)"] = cuadro_rrc_fmt["Dif. vs Exacto (MXN)"].map("${:+,.2f}".format)
cuadro_rrc_fmt["Dif. vs Exacto (%)"] = cuadro_rrc_fmt["Dif. vs Exacto (%)"].map("{:+.2f}%".format)
cuadro_rrc_fmt["RRC sobre Prima Emitida (MXN)"] = cuadro_rrc_fmt["RRC sobre Prima Emitida (MXN)"].map("${:,.2f}".format)
display(cuadro_rrc_fmt)

# Gráfica comparativa de RRC
fig_rrc = px.bar(
    cuadro_rrc,
    x="Método",
    y="RRC sobre Prima de Tarifa (MXN)",
    color="Método",
    text=cuadro_rrc["RRC sobre Prima de Tarifa (MXN)"].map(lambda v: f"${v/1e6:,.1f} M"),
    color_discrete_sequence=["#636EFA", "#EF553B", "#00CC96", "#AB63FA"],
    title="Comparación de la Reserva de Riesgos en Curso (RRC) al 31/12/2025",
)
fig_rrc.update_traces(textposition="outside")
fig_rrc.update_layout(
    template="plotly",
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",
    font=dict(family="Arial", size=13, color="#2A4163"),
    showlegend=False,
    height=500,
    yaxis=dict(title="Monto de Reserva (MXN)", tickformat=",.0f", range=[0, rrc_24avos_tarifa * 1.18]),
    xaxis=dict(title=""),
)
fig_rrc.show()

,Método,Criterio / Fórmula,RRC sobre Prima de Tarifa (MXN),% de Prima de Tarifa,Dif. vs Exacto (MXN),Dif. vs Exacto (%),RRC sobre Prima Emitida (MXN)
0,1. Método Exacto (Póliza por póliza),Σ Prima_Tarifa_k × (Días por devengar_k / Dura...,"$5,244,839,068.74",86.28%,$+0.00,+0.00%,"$5,282,060,815.21"
1,2. Método de los Veinticuatroavos,Σ Prima_Tarifa_m × (2m - 1) / 24,"$5,284,812,484.50",86.94%,"$+39,973,415.76",+0.76%,"$5,322,232,615.25"
2,3a. Método Estatutario (S̄ 2015–2022 = 65.08%),PND_Exacto × S̄_2015-2022 (años con desarrollo...,"$3,413,597,990.56",56.16%,"$-1,831,241,078.18",-34.92%,"$3,437,823,725.09"
3,3b. Método Estatutario (S̄ 2015–2025 = 61.18%),PND_Exacto × S̄_2015-2025 (todos los años obse...,"$3,208,653,187.09",52.79%,"$-2,036,185,881.64",-38.82%,"$3,231,424,462.61"


Figure({
    'data': [{'hovertemplate': ('Método=%{x}<br>RRC sobre Prima' ... 'br>text=%{text}<extra></extra>'),
              'legendgroup': '1. Método Exacto (Póliza por póliza)',
              'marker': {'color': '#636EFA', 'pattern': {'shape': ''}},
              'name': '1. Método Exacto (Póliza por póliza)',
              'orientation': 'v',
              'showlegend': True,
              'text': array(['$5,244.8 M'], dtype=object),
              'textposition': 'outside',
              'type': 'bar',
              'x': array(['1. Método Exacto (Póliza por póliza)'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'TcTLSd6J80E=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Método=%{x}<br>RRC sobre Prima' ... 'br>text=%{text}<extra></extra>'),
              'legendgroup': '2. Método de los Veinticuatroavos',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': '2. Método de los Veint

### Interpretación de los resultados de la RRC

1. **Elevada proporción de prima no devengada ($86.28\%$ en el método exacto y $86.94\%$ en veinticuatroavos):**
   - En diciembre de 2025 se concentró el **$79.99\%$ de la emisión anual** ($\$4,920.59$ millones de prima emitida, equivalente a $\$4,914.17$ millones de prima de tarifa). Dado que las pólizas emitidas en diciembre apenas han transcurrido unos días al 31 de diciembre de 2025, la mayor parte de su vigencia se devengará durante 2026.
2. **Cercanía entre el Método Exacto ($\$5,244.84\text{ M}$) y el de Veinticuatroavos ($\$5,284.81\text{ M}$):**
   - El método de los veinticuatroavos sobreestima ligeramente la reserva en **$+\$39.97\text{ millones}$ ($+0.76\%$)** respecto al método exacto. Esto ocurre porque el supuesto de emisión en el día 15 de cada mes asigna en diciembre una fracción no devengada de $23/24 = 95.833\%$, ligeramente superior al promedio exacto de días por devengar de las pólizas suscritas en dicho mes, además de aplicar la regla anual a los 87 endosos/cancelaciones de vigencia menor a un año.
3. **Contraste con el Método Estatutario ($\$3,413.60\text{ M}$ con $\bar{S}_{2015\text{–}2022}$ y $\$3,208.65\text{ M}$ con $\bar{S}_{2015\text{–}2025}$):**
   - Mientras los métodos exacto y de veinticuatroavos miden el **pasivo contable por prima no devengada (PND)** al $100\%$ de la prima de tarifa, el método estatutario estima el **valor esperado de las obligaciones futuras por siniestros ($\text{BEL} = \text{PND} \times \bar{S}$)**. Como la siniestralidad media del mercado es inferior a $1$ ($65.08\%$ en años maduros), la obligación esperada por siniestralidad futura es **$34.92\%$ menor** ($-\$1,831.24\text{ M}$) que la prima de tarifa no devengada.

---
# 6. Reserva de Siniestros Ocurridos y No Reportados (SONR) al 31 de diciembre de 2025

La **Reserva de Obligaciones Pendientes de Cumplir por Siniestros Ocurridos y No Reportados (SONR / IBNR)** estima los pagos futuros que habrán de realizarse después del 31 de diciembre de 2025 por siniestros cuyo año de origen fue 2025 o anterior.

## 6.1. Construcción y Validación de los Triángulos de Desarrollo (Mercado Total)

En la base `siniestros_ajustados_2025.parquet`, cada registro cuenta con cuatro columnas de desarrollo (`0_monto pagado_ajustada` a `3_monto pagado_ajustada`), actualizadas por INPC según el año efectivo de pago ($\text{año de origen} + j \le 2025$):
- Las columnas originales registran **pagos incrementales** $X_{i,j}$ realizados en el año de desarrollo $j \in \{0, 1, 2, 3\}$ para siniestros ocurridos en el año de origen $i \in \{2015, \dots, 2025\}$.
- Para los años de origen **2015 a 2022**, se conocen los cuatro desarrollos completos ($j = 0, 1, 2, 3$).
- Para **2023** falta observar $j = 3$ (pagos en 2026); para **2024** faltan $j = 2, 3$ (pagos en 2026 y 2027); y para **2025** faltan $j = 1, 2, 3$ (pagos en 2026, 2027 y 2028).
- El **triángulo acumulado** se define como:
  $$C_{i,j} = \sum_{k=0}^{j} X_{i,k}, \qquad j \in \{0, 1, 2, 3\}$$

In [7]:
# ------------------------------------------
# 6.1 Construcción de triángulos incremental y acumulado
# ------------------------------------------
tri_incremental, tri_acumulado = construir_triangulos_desarrollo(
    siniestros_ajustados,
    columnas_desarrollo=columnas_pagos_ajustados,
    col_anio="año",
)

print("Triángulo de Pagos Incrementales X_{i,j} (MXN a precios de dic. 2025):")
display(tri_incremental.map(lambda v: "—" if pd.isna(v) else f"${v:,.2f}"))

print("Triángulo de Pagos Acumulados C_{i,j} (MXN a precios de dic. 2025):")
display(tri_acumulado.map(lambda v: "—" if pd.isna(v) else f"${v:,.2f}"))

Triángulo de Pagos Incrementales X_{i,j} (MXN a precios de dic. 2025):
Triángulo de Pagos Acumulados C_{i,j} (MXN a precios de dic. 2025):


,0,1,2,3
año,,,,
2015,"$4,248,743,261.24","$26,207,761.12","$8,962,783.13","$1,655,224.62"
2016,"$877,634,083.39","$188,580,204.22","$383,471,811.37","$29,222,408.47"
2017,"$972,017,277.95","$40,134,925.97","$4,623,780.19","$4,777,467.97"
2018,"$677,616,103.39","$246,513,523.52","$31,429,935.42","$3,948,570.59"
2019,"$1,271,501,565.42","$49,063,357.56","$34,231,267.76","$35,475,140.61"
2020,"$884,580,469.88","$127,759,871.10","$43,629,103.14","$14,878,568.42"
2021,"$859,741,647.30","$65,896,146.76","$83,194,419.96","$8,070,205.27"
2022,"$1,069,098,378.02","$228,280,461.22","$7,417,511.14","$17,337,726.20"
2023,"$1,026,272,998.50","$736,927,192.04","$275,744,260.74",—


,0,1,2,3
año,,,,
2015,"$4,248,743,261.24","$4,274,951,022.36","$4,283,913,805.48","$4,285,569,030.11"
2016,"$877,634,083.39","$1,066,214,287.61","$1,449,686,098.98","$1,478,908,507.45"
2017,"$972,017,277.95","$1,012,152,203.92","$1,016,775,984.11","$1,021,553,452.08"
2018,"$677,616,103.39","$924,129,626.91","$955,559,562.33","$959,508,132.92"
2019,"$1,271,501,565.42","$1,320,564,922.98","$1,354,796,190.75","$1,390,271,331.35"
2020,"$884,580,469.88","$1,012,340,340.99","$1,055,969,444.13","$1,070,848,012.55"
2021,"$859,741,647.30","$925,637,794.06","$1,008,832,214.02","$1,016,902,419.30"
2022,"$1,069,098,378.02","$1,297,378,839.24","$1,304,796,350.39","$1,322,134,076.59"
2023,"$1,026,272,998.50","$1,763,200,190.54","$2,038,944,451.28",—


## 6.2. Método Chain-Ladder (Mercado Total — sin conglomerados)

El método **Chain-Ladder** proyecta los pagos acumulados futuros suponiendo que el desarrollo relativo entre los periodos $j$ y $j+1$ mantiene un patrón proporcional estable a través de los años de origen.

1. **Factores de desarrollo individuales (*link ratios*):**
   $$f_{i,j} = \frac{C_{i,j+1}}{C_{i,j}}, \qquad j \in \{0, 1, 2\}$$
2. **Selección de factores de desarrollo $\hat{f}_j$:**
   - **Promedio ponderado por volumen (estándar Chain-Ladder):**
     $$\hat{f}_j^{\text{pond}} = \frac{\sum_{i=2015}^{2024-j} C_{i,j+1}}{\sum_{i=2015}^{2024-j} C_{i,j}}$$
   - **Promedio aritmético simple (variante equiponderada):**
     $$\bar{f}_j^{\text{simple}} = \frac{1}{n_j} \sum_{i=2015}^{2024-j} \frac{C_{i,j+1}}{C_{i,j}}$$
3. **Proyección al siniestro último (*Ultimate*) y cálculo de la SONR:**
   Para cada año de origen $i$ con último desarrollo observado $j^*(i) = 2025 - i < 3$:
   $$\hat{C}_{i,3} = C_{i, j^*(i)} \times \prod_{j=j^*(i)}^{2} \hat{f}_j, \qquad \text{SONR}_i = \hat{C}_{i,3} - C_{i, j^*(i)}$$

In [8]:
# ------------------------------------------
# 6.2 Cálculo de SONR por Chain-Ladder (Mercado Total)
# ------------------------------------------
# Matriz de factores individuales f_{i,j} = C_{i,j+1} / C_{i,j}
factores_individuales = pd.DataFrame(index=tri_acumulado.index)
for j in range(3):
    factores_individuales[f"Desarrollo {j} → {j+1}"] = tri_acumulado[j + 1] / tri_acumulado[j]

res_cl_pond = calcular_sonr_chain_ladder(tri_acumulado, metodo_factores="ponderado")
res_cl_simp = calcular_sonr_chain_ladder(tri_acumulado, metodo_factores="simple")

# Añadir filas resumen de factores seleccionados
resumen_factores_cl = pd.DataFrame(
    [res_cl_pond["factores"], res_cl_simp["factores"]],
    index=["Factor Ponderado por Volumen (f̂_j)", "Factor Promedio Simple (f̄_j)"],
    columns=factores_individuales.columns,
)
factores_mostrar = pd.concat([factores_individuales.dropna(how="all"), resumen_factores_cl])

print("Factores de desarrollo individuales y promedios seleccionados:")
display(factores_mostrar.map(lambda v: "—" if pd.isna(v) else f"{v:.6f}"))

print("Triángulo Acumulado Proyectado (Chain-Ladder Ponderado por Volumen):")
display(res_cl_pond["triangulo_proyectado"].map(lambda v: f"${v:,.2f}"))

# Resumen anual para los años con desarrollo pendiente (2023, 2024, 2025)
tabla_cl_anual = pd.DataFrame({
    "Pagado Acumulado Actual (C_{i, 2025-i})": res_cl_pond["resumen_anual"]["pagado_acumulado"],
    "Factor Acumulado al Ultimate (Pond.)": [
        1.0 if i <= 2022 else np.prod(res_cl_pond["factores"][2025 - i:])
        for i in tri_acumulado.index
    ],
    "Ultimate CL Ponderado": res_cl_pond["resumen_anual"]["siniestro_ultimo"],
    "SONR CL Ponderado": res_cl_pond["resumen_anual"]["sonr"],
    "Ultimate CL Simple": res_cl_simp["resumen_anual"]["siniestro_ultimo"],
    "SONR CL Simple": res_cl_simp["resumen_anual"]["sonr"],
}).loc[2023:2025]

tabla_cl_anual.loc["Total SONR (2023–2025)"] = [
    tabla_cl_anual["Pagado Acumulado Actual (C_{i, 2025-i})"].sum(),
    np.nan,
    tabla_cl_anual["Ultimate CL Ponderado"].sum(),
    tabla_cl_anual["SONR CL Ponderado"].sum(),
    tabla_cl_anual["Ultimate CL Simple"].sum(),
    tabla_cl_anual["SONR CL Simple"].sum(),
]

sonr_cl_pond_total = float(res_cl_pond["sonr_total"])
sonr_cl_simp_total = float(res_cl_simp["sonr_total"])

print(f"SONR Total Chain-Ladder (Ponderado por Volumen): ${sonr_cl_pond_total:,.2f}")
print(f"SONR Total Chain-Ladder (Promedio Simple):       ${sonr_cl_simp_total:,.2f}")

tabla_cl_fmt = tabla_cl_anual.copy()
for col in tabla_cl_fmt.columns:
    if "Factor" in col:
        tabla_cl_fmt[col] = tabla_cl_fmt[col].map(lambda v: "—" if pd.isna(v) else f"{v:.6f}")
    else:
        tabla_cl_fmt[col] = tabla_cl_fmt[col].map(lambda v: f"${v:,.2f}")
display(tabla_cl_fmt)

Factores de desarrollo individuales y promedios seleccionados:
Triángulo Acumulado Proyectado (Chain-Ladder Ponderado por Volumen):
SONR Total Chain-Ladder (Ponderado por Volumen): $704,845,155.68
SONR Total Chain-Ladder (Promedio Simple):       $924,972,814.42


,Desarrollo 0 → 1,Desarrollo 1 → 2,Desarrollo 2 → 3
2015,1.006168,1.002097,1.000386
2016,1.214873,1.359657,1.020158
2017,1.041290,1.004568,1.004699
2018,1.363795,1.034010,1.004132
2019,1.038587,1.025922,1.026185
2020,1.144430,1.043097,1.014090
2021,1.076646,1.089878,1.008000
2022,1.213526,1.005717,1.013288
2023,1.718062,1.156389,—
2024,1.141878,—,—


,0,1,2,3
año,,,,
2015,"$4,248,743,261.24","$4,274,951,022.36","$4,283,913,805.48","$4,285,569,030.11"
2016,"$877,634,083.39","$1,066,214,287.61","$1,449,686,098.98","$1,478,908,507.45"
2017,"$972,017,277.95","$1,012,152,203.92","$1,016,775,984.11","$1,021,553,452.08"
2018,"$677,616,103.39","$924,129,626.91","$955,559,562.33","$959,508,132.92"
2019,"$1,271,501,565.42","$1,320,564,922.98","$1,354,796,190.75","$1,390,271,331.35"
2020,"$884,580,469.88","$1,012,340,340.99","$1,055,969,444.13","$1,070,848,012.55"
2021,"$859,741,647.30","$925,637,794.06","$1,008,832,214.02","$1,016,902,419.30"
2022,"$1,069,098,378.02","$1,297,378,839.24","$1,304,796,350.39","$1,322,134,076.59"
2023,"$1,026,272,998.50","$1,763,200,190.54","$2,038,944,451.28","$2,057,867,800.03"


,"Pagado Acumulado Actual (C_{i, 2025-i})",Factor Acumulado al Ultimate (Pond.),Ultimate CL Ponderado,SONR CL Ponderado,Ultimate CL Simple,SONR CL Simple
año,,,,,,
2023,"$2,038,944,451.28",1.009281,"$2,057,867,800.03","$18,923,348.75","$2,062,121,396.90","$23,176,945.62"
2024,"$2,767,550,968.37",1.074062,"$2,972,522,228.59","$204,971,260.23","$3,023,346,142.78","$255,795,174.42"
2025,"$2,107,938,544.00",1.228162,"$2,588,889,090.70","$480,950,546.70","$2,753,939,238.39","$646,000,694.39"
Total SONR (2023–2025),"$6,914,433,963.65",—,"$7,619,279,119.33","$704,845,155.68","$7,839,406,778.07","$924,972,814.42"


### 6.2.1. Chain-Ladder por Conglomerados *(Apartado Pendiente)*

> **Nota metodológica:** El cálculo de la SONR desagregando los triángulos de desarrollo para cada uno de los conglomerados obtenidos en el apartado 4 (`03_siniestralidad_conglomerados.ipynb`) se deja **pendiente** en esta entrega para integrarse por separado. Todo el resto de las estimaciones en este cuaderno corresponde al **mercado total agregado (sin conglomerados)**.

## 6.3. Método de la Razón (Razón de Siniestralidad Esperada)

El **método de la razón** (*Expected Loss Ratio* / razón de siniestralidad) estima las obligaciones totales de cada año de origen a partir de la exposición medida por la **prima emitida** y la **razón de siniestralidad histórica media del mercado** ($\bar{S}$):

$$\text{Siniestro Último Esperado}_i = \text{Prima Emitida}_i \times \bar{S}$$

Se presentan las dos formulaciones estándar vistas en la práctica actuarial:

1. **Razón ponderada por maduración (Bornhuetter-Ferguson / proporción pendiente de reportar):**
   Aplica la siniestralidad esperada $\text{Prima}_i \times \bar{S}$ a la fracción aún no reportada al cierre de 2025 según el patrón de desarrollo acumulado $\left(1 - \frac{1}{F_{2025-i}}\right)$:
   $$\text{SONR}_i^{\text{Razón-BF}} = \text{Prima Emitida}_i \times \bar{S} \times \left(1 - \frac{1}{F_{2025-i}}\right)$$
   Esta formulación evita que un año con siniestralidad pagada atípicamente alta en el desarrollo temprano genere reservas negativas.

2. **Razón de Siniestralidad Directa (*Expected Loss Ratio* puro):**
   Obtiene la reserva como la diferencia directa entre el siniestro último esperado por razón de siniestralidad y el monto acumulado ya pagado al 31 de diciembre de 2025:
   $$\text{SONR}_i^{\text{Razón Directa}} = \max\left(0,\; \text{Prima Emitida}_i \times \bar{S} - C_{i, 2025-i}\right)$$
   *(Para el ejercicio 2025 en particular: $\text{SONR}_{2025} = \text{Prima}_{2025} \times \bar{S} - C_{2025, 0}$).*

In [9]:
# ------------------------------------------
# 6.3 Cálculo de la SONR por el Método de la Razón
# ------------------------------------------
anios_incompletos = [2023, 2024, 2025]
primas_inc = primas_anual.loc[anios_incompletos]
pagado_inc = res_cl_pond["resumen_anual"].loc[anios_incompletos, "pagado_acumulado"]

# Factores acumulados al ultimate F_{2025-i}
f_01, f_12, f_23 = res_cl_pond["factores"]
factores_ult_inc = pd.Series(
    [f_23, f_12 * f_23, f_01 * f_12 * f_23],
    index=anios_incompletos,
    name="F_ult",
)
pct_no_reportado = 1.0 - (1.0 / factores_ult_inc)

# Cálculo con S̄_2015-2022 = 65.0849% (años con desarrollo completo) y S̄_2015-2025 = 61.1773%
ult_esperado_15_22 = primas_inc * s_media_2015_2022
ult_esperado_15_25 = primas_inc * s_media_2015_2025

# 1) Razón ponderada por desarrollo pendiente (Bornhuetter-Ferguson)
sonr_razon_bf_15_22 = ult_esperado_15_22 * pct_no_reportado
sonr_razon_bf_15_25 = ult_esperado_15_25 * pct_no_reportado

# 2) Razón directa: Prima * S̄ - Pagado (truncando en 0 cuando Pagado > Esperado, como en 2024)
dif_directa_15_22 = ult_esperado_15_22 - pagado_inc
dif_directa_15_25 = ult_esperado_15_25 - pagado_inc
sonr_razon_dir_15_22 = dif_directa_15_22.clip(lower=0.0)
sonr_razon_dir_15_25 = dif_directa_15_25.clip(lower=0.0)

# Solo año 2025 bajo razón directa (Prima_2025 * S̄ - Pagado_2025)
sonr_razon_solo_2025_15_22 = float(sonr_razon_dir_15_22.loc[2025])
sonr_razon_solo_2025_15_25 = float(sonr_razon_dir_15_25.loc[2025])

tabla_razon = pd.DataFrame({
    "Prima Emitida Ajustada": primas_inc,
    "Pagado Acumulado Actual": pagado_inc,
    "Siniestro Esperado (S̄ 65.08%)": ult_esperado_15_22,
    "% Pendiente (1 - 1/F)": pct_no_reportado * 100,
    "SONR Razón Ponderada / BF (S̄ 65.08%)": sonr_razon_bf_15_22,
    "SONR Razón Ponderada / BF (S̄ 61.18%)": sonr_razon_bf_15_25,
    "Dif. Directa (Prima×S̄_65.08% - Pagado)": dif_directa_15_22,
    "SONR Razón Directa Truncada (S̄ 65.08%)": sonr_razon_dir_15_22,
})

tabla_razon.loc["Total (2023–2025)"] = [
    primas_inc.sum(),
    pagado_inc.sum(),
    ult_esperado_15_22.sum(),
    np.nan,
    sonr_razon_bf_15_22.sum(),
    sonr_razon_bf_15_25.sum(),
    dif_directa_15_22.sum(),
    sonr_razon_dir_15_22.sum(),
]

print(f"SONR Método de la Razón Ponderada / BF (S̄ 2015–2022 = 65.08%): ${sonr_razon_bf_15_22.sum():,.2f}")
print(f"SONR Método de la Razón Ponderada / BF (S̄ 2015–2025 = 61.18%): ${sonr_razon_bf_15_25.sum():,.2f}")
print(f"SONR Método de la Razón Directa solo 2025 (S̄ 2015–2022):       ${sonr_razon_solo_2025_15_22:,.2f}")
print(f"SONR Método de la Razón Directa 2023–2025 truncada (S̄ 65.08%): ${sonr_razon_dir_15_22.sum():,.2f}")

tabla_razon_fmt = tabla_razon.copy()
for col in tabla_razon_fmt.columns:
    if "%" in col:
        tabla_razon_fmt[col] = tabla_razon_fmt[col].map(lambda v: "—" if pd.isna(v) else f"{v:.2f}%")
    elif "Dif." in col:
        tabla_razon_fmt[col] = tabla_razon_fmt[col].map(lambda v: f"${v:+,.2f}")
    else:
        tabla_razon_fmt[col] = tabla_razon_fmt[col].map(lambda v: f"${v:,.2f}")
display(tabla_razon_fmt)

SONR Método de la Razón Ponderada / BF (S̄ 2015–2022 = 65.08%): $939,837,508.06
SONR Método de la Razón Ponderada / BF (S̄ 2015–2025 = 61.18%): $883,411,761.99
SONR Método de la Razón Directa solo 2025 (S̄ 2015–2022):       $1,895,721,800.80
SONR Método de la Razón Directa 2023–2025 truncada (S̄ 65.08%): $2,852,695,652.72


,Prima Emitida Ajustada,Pagado Acumulado Actual,Siniestro Esperado (S̄ 65.08%),% Pendiente (1 - 1/F),SONR Razón Ponderada / BF (S̄ 65.08%),SONR Razón Ponderada / BF (S̄ 61.18%),Dif. Directa (Prima×S̄_65.08% - Pagado),SONR Razón Directa Truncada (S̄ 65.08%)
2023,"$4,603,093,102.01","$2,038,944,451.28",2995918303.20%,0.92%,27549294.90%,25895296.73%,956973851.92%,956973851.92%
2024,"$3,754,691,817.93","$2,767,550,968.37",2443737219.93%,6.90%,168508713.85%,158391826.82%,-323813748.43%,0.00%
2025,"$6,151,443,214.00","$2,107,938,544.00",4003660344.80%,18.58%,743779499.31%,699124638.45%,1895721800.80%,1895721800.80%
Total (2023–2025),"$14,509,228,133.94","$6,914,433,963.65",9443315867.94%,—,939837508.06%,883411761.99%,2528881904.29%,2852695652.72%


## 6.4. Método Estatutario para la SONR

El **método estatutario** para la SONR estima las obligaciones ocurridas y no reportadas aplicando factores históricos/estatutarios de siniestralidad diferida sobre las primas emitidas (o siniestros pagados en el año de origen).

A partir de los ocho años con desarrollo completo (**2015–2022**), definimos para cada año de origen $i$ y cada año de desarrollo $j \in \{0, 1, 2, 3\}$ el factor de pago incremental respecto a la prima emitida:

$$r_{i,j} = \frac{X_{i,j}}{\text{Prima Emitida}_i}, \qquad \bar{r}_j = \frac{1}{8}\sum_{i=2015}^{2022} r_{i,j}$$

Obsérvese la identidad exacta entre los factores incrementales por desarrollo y el factor de siniestralidad total promedio:
$$\bar{r}_0 + \bar{r}_1 + \bar{r}_2 + \bar{r}_3 = 0.560919 + 0.053814 + 0.030329 + 0.005787 = 0.650849 = \bar{S}_{2015\text{–}2022}$$

De aquí que la proporción histórica media de siniestros que se pagan **en años posteriores al año de ocurrencia** ($j = 1, 2, 3$) respecto a la prima emitida es:
$$\bar{\phi}_{\text{prima}} = \bar{r}_1 + \bar{r}_2 + \bar{r}_3 = 8.9930\%$$

Bajo este criterio estatutario se obtienen:
1. **SONR Estatutaria acumulada por rezagos pendientes (2023–2025):**
   - Para 2023 (pendiente $j=3$): $\text{Prima}_{2023} \times \bar{r}_3$
   - Para 2024 (pendientes $j=2, 3$): $\text{Prima}_{2024} \times (\bar{r}_2 + \bar{r}_3)$
   - Para 2025 (pendientes $j=1, 2, 3$): $\text{Prima}_{2025} \times (\bar{r}_1 + \bar{r}_2 + \bar{r}_3)$
2. **SONR Estatutaria del ejercicio 2025 sobre Prima Emitida:** $\text{Prima}_{2025} \times \bar{\phi}_{\text{prima}} = \text{Prima}_{2025} \times 8.9930\%$.
3. **SONR Estatutaria sobre Siniestros del ejercicio 2025:** aplicando la proporción media histórica de pagos posteriores respecto al pago del año 0 ($\bar{\phi}_{\text{sin}} = \text{media}\left(\frac{X_{i,1}+X_{i,2}+X_{i,3}}{X_{i,0}}\right) = 23.5526\%$) al monto pagado en 2025.

In [10]:
# ------------------------------------------
# 6.4 Cálculo de la SONR por el Método Estatutario
# ------------------------------------------
# Factores incrementales respecto a la prima emitida r_{i,j} = X_{i,j} / Prima_i
ratios_inc_prima = tri_incremental.div(primas_anual, axis=0)
r_bar_2015_2022 = ratios_inc_prima.loc[2015:2022].mean()
r_bar_todos = ratios_inc_prima.mean()

phi_prima_15_22 = float(r_bar_2015_2022.loc[1:3].sum())  # 8.9930%

# Proporción de pagos posteriores (j=1,2,3) respecto al pago en año de origen (j=0) en 2015-2022
pagos_posteriores_15_22 = tri_incremental.loc[2015:2022, [1, 2, 3]].sum(axis=1)
phi_siniestros_15_22 = float((pagos_posteriores_15_22 / tri_incremental.loc[2015:2022, 0]).mean())

# 1) SONR Estatutaria por rezagos pendientes (2023-2025) sobre Prima Emitida
sonr_est_2023 = float(primas_anual.loc[2023] * r_bar_2015_2022[3])
sonr_est_2024 = float(primas_anual.loc[2024] * (r_bar_2015_2022[2] + r_bar_2015_2022[3]))
sonr_est_2025 = float(primas_anual.loc[2025] * (r_bar_2015_2022[1] + r_bar_2015_2022[2] + r_bar_2015_2022[3]))
sonr_estatutario_rezagos = sonr_est_2023 + sonr_est_2024 + sonr_est_2025

# 2) SONR Estatutaria sobre Prima Emitida del ejercicio 2025 (Prima_2025 * 8.9930%)
sonr_estatutario_prima_2025 = sonr_est_2025

# 3) SONR Estatutaria sobre Siniestros Pagados en 2025 (Pagado_2025 * 23.5526%)
sonr_estatutario_sin_2025 = float(tri_incremental.loc[2025, 0] * phi_siniestros_15_22)

resumen_tasas_est = pd.DataFrame({
    "Desarrollo j": ["j = 0 (Año de ocurrencia)", "j = 1 (+1 año)", "j = 2 (+2 años)", "j = 3 (+3 años)", "Suma desarrollos posteriores (j = 1..3)"],
    "Proporción Media s/Prima (2015–2022)": [
        f"{r_bar_2015_2022[0] * 100:.4f}%",
        f"{r_bar_2015_2022[1] * 100:.4f}%",
        f"{r_bar_2015_2022[2] * 100:.4f}%",
        f"{r_bar_2015_2022[3] * 100:.4f}%",
        f"{phi_prima_15_22 * 100:.4f}%",
    ],
    "Proporción Media s/Prima (Todos disp.)": [
        f"{r_bar_todos[0] * 100:.4f}%",
        f"{r_bar_todos[1] * 100:.4f}%",
        f"{r_bar_todos[2] * 100:.4f}%",
        f"{r_bar_todos[3] * 100:.4f}%",
        f"{r_bar_todos.loc[1:3].sum() * 100:.4f}%",
    ],
})
display(resumen_tasas_est)

print(f"SONR Estatutario (Rezagos pendientes 2023–2025 s/Prima): ${sonr_estatutario_rezagos:,.2f}")
print(f"  - Año 2023 (falta j=3, 0.5787%):                       ${sonr_est_2023:,.2f}")
print(f"  - Año 2024 (faltan j=2..3, 3.6116%):                   ${sonr_est_2024:,.2f}")
print(f"  - Año 2025 (faltan j=1..3, 8.9930%):                   ${sonr_est_2025:,.2f}")
print(f"SONR Estatutario (Factor 23.55% s/Siniestros 2025):      ${sonr_estatutario_sin_2025:,.2f}")

SONR Estatutario (Rezagos pendientes 2023–2025 s/Prima): $715,437,491.59
  - Año 2023 (falta j=3, 0.5787%):                       $26,635,967.59
  - Año 2024 (faltan j=2..3, 3.6116%):                   $135,602,791.16
  - Año 2025 (faltan j=1..3, 8.9930%):                   $553,198,732.83
SONR Estatutario (Factor 23.55% s/Siniestros 2025):      $496,473,701.45


,Desarrollo j,Proporción Media s/Prima (2015–2022),Proporción Media s/Prima (Todos disp.)
0,j = 0 (Año de ocurrencia),56.0919%,51.8044%
1,j = 1 (+1 año),5.3814%,6.8219%
2,j = 2 (+2 años),3.0329%,3.3615%
3,j = 3 (+3 años),0.5787%,0.5787%
4,Suma desarrollos posteriores (j = 1..3),8.9930%,10.7621%


## 6.5. Cuadro Comparativo de la Reserva SONR al 31 de diciembre de 2025

En la siguiente tabla se resumen y comparan los resultados obtenidos para la **SONR al 31 de diciembre de 2025** para el mercado total bajo los tres métodos solicitados (**Chain-Ladder**, **Método de la Razón** y **Método Estatutario**), tomando como referencia base el método **Chain-Ladder Ponderado por Volumen** ($\$704,845,155.68$) para calcular las diferencias absolutas y porcentuales.

In [11]:
# ------------------------------------------
# 6.5 Cuadro comparativo de SONR al 31/12/2025
# ------------------------------------------
base_sonr = sonr_cl_pond_total

cuadro_sonr = pd.DataFrame([
    {
        "Método": "1a. Chain-Ladder (Ponderado por Volumen)",
        "Fórmula / Criterio": "Triángulo acumulado con factores f̂_j = ΣC_{i,j+1} / ΣC_{i,j}",
        "Año 2023 (MXN)": float(res_cl_pond["resumen_anual"].loc[2023, "sonr"]),
        "Año 2024 (MXN)": float(res_cl_pond["resumen_anual"].loc[2024, "sonr"]),
        "Año 2025 (MXN)": float(res_cl_pond["resumen_anual"].loc[2025, "sonr"]),
        "SONR Total Mercado (MXN)": sonr_cl_pond_total,
        "Dif. vs CL Pond. (MXN)": 0.0,
        "Dif. vs CL Pond. (%)": 0.0,
    },
    {
        "Método": "1b. Chain-Ladder (Promedio Simple)",
        "Fórmula / Criterio": "Triángulo acumulado con factores f̄_j = media(C_{i,j+1} / C_{i,j})",
        "Año 2023 (MXN)": float(res_cl_simp["resumen_anual"].loc[2023, "sonr"]),
        "Año 2024 (MXN)": float(res_cl_simp["resumen_anual"].loc[2024, "sonr"]),
        "Año 2025 (MXN)": float(res_cl_simp["resumen_anual"].loc[2025, "sonr"]),
        "SONR Total Mercado (MXN)": sonr_cl_simp_total,
        "Dif. vs CL Pond. (MXN)": sonr_cl_simp_total - base_sonr,
        "Dif. vs CL Pond. (%)": ((sonr_cl_simp_total / base_sonr) - 1) * 100,
    },
    {
        "Método": "2a. Método de la Razón Ponderada / BF (S̄ 2015–2022 = 65.08%)",
        "Fórmula / Criterio": "Σ Prima_i × S̄_2015-2022 × (1 - 1/F_i)",
        "Año 2023 (MXN)": float(sonr_razon_bf_15_22.loc[2023]),
        "Año 2024 (MXN)": float(sonr_razon_bf_15_22.loc[2024]),
        "Año 2025 (MXN)": float(sonr_razon_bf_15_22.loc[2025]),
        "SONR Total Mercado (MXN)": float(sonr_razon_bf_15_22.sum()),
        "Dif. vs CL Pond. (MXN)": float(sonr_razon_bf_15_22.sum()) - base_sonr,
        "Dif. vs CL Pond. (%)": ((float(sonr_razon_bf_15_22.sum()) / base_sonr) - 1) * 100,
    },
    {
        "Método": "2b. Método de la Razón Ponderada / BF (S̄ 2015–2025 = 61.18%)",
        "Fórmula / Criterio": "Σ Prima_i × S̄_2015-2025 × (1 - 1/F_i)",
        "Año 2023 (MXN)": float(sonr_razon_bf_15_25.loc[2023]),
        "Año 2024 (MXN)": float(sonr_razon_bf_15_25.loc[2024]),
        "Año 2025 (MXN)": float(sonr_razon_bf_15_25.loc[2025]),
        "SONR Total Mercado (MXN)": float(sonr_razon_bf_15_25.sum()),
        "Dif. vs CL Pond. (MXN)": float(sonr_razon_bf_15_25.sum()) - base_sonr,
        "Dif. vs CL Pond. (%)": ((float(sonr_razon_bf_15_25.sum()) / base_sonr) - 1) * 100,
    },
    {
        "Método": "2c. Método de la Razón Directa — Ejercicio 2025 (S̄ = 65.08%)",
        "Fórmula / Criterio": "Prima_2025 × S̄_2015-2022 - Pagado_2025",
        "Año 2023 (MXN)": 0.0,
        "Año 2024 (MXN)": 0.0,
        "Año 2025 (MXN)": sonr_razon_solo_2025_15_22,
        "SONR Total Mercado (MXN)": sonr_razon_solo_2025_15_22,
        "Dif. vs CL Pond. (MXN)": sonr_razon_solo_2025_15_22 - base_sonr,
        "Dif. vs CL Pond. (%)": ((sonr_razon_solo_2025_15_22 / base_sonr) - 1) * 100,
    },
    {
        "Método": "3a. Método Estatutario por Rezagos s/Prima (2023–2025)",
        "Fórmula / Criterio": "Σ Prima_i × Σ_{j > 2025-i} r̄_j (tasas medias 2015–2022)",
        "Año 2023 (MXN)": sonr_est_2023,
        "Año 2024 (MXN)": sonr_est_2024,
        "Año 2025 (MXN)": sonr_est_2025,
        "SONR Total Mercado (MXN)": sonr_estatutario_rezagos,
        "Dif. vs CL Pond. (MXN)": sonr_estatutario_rezagos - base_sonr,
        "Dif. vs CL Pond. (%)": ((sonr_estatutario_rezagos / base_sonr) - 1) * 100,
    },
    {
        "Método": "3b. Método Estatutario Ejercicio 2025 s/Prima (8.99%)",
        "Fórmula / Criterio": "Prima_2025 × φ̄_prima (8.9930%)",
        "Año 2023 (MXN)": 0.0,
        "Año 2024 (MXN)": 0.0,
        "Año 2025 (MXN)": sonr_estatutario_prima_2025,
        "SONR Total Mercado (MXN)": sonr_estatutario_prima_2025,
        "Dif. vs CL Pond. (MXN)": sonr_estatutario_prima_2025 - base_sonr,
        "Dif. vs CL Pond. (%)": ((sonr_estatutario_prima_2025 / base_sonr) - 1) * 100,
    },
    {
        "Método": "3c. Método Estatutario Ejercicio 2025 s/Siniestros (23.55%)",
        "Fórmula / Criterio": "Pagado_2025 × φ̄_sin (23.5526%)",
        "Año 2023 (MXN)": 0.0,
        "Año 2024 (MXN)": 0.0,
        "Año 2025 (MXN)": sonr_estatutario_sin_2025,
        "SONR Total Mercado (MXN)": sonr_estatutario_sin_2025,
        "Dif. vs CL Pond. (MXN)": sonr_estatutario_sin_2025 - base_sonr,
        "Dif. vs CL Pond. (%)": ((sonr_estatutario_sin_2025 / base_sonr) - 1) * 100,
    },
])

cuadro_sonr_fmt = cuadro_sonr.copy()
for col in ["Año 2023 (MXN)", "Año 2024 (MXN)", "Año 2025 (MXN)", "SONR Total Mercado (MXN)"]:
    cuadro_sonr_fmt[col] = cuadro_sonr_fmt[col].map("${:,.2f}".format)
cuadro_sonr_fmt["Dif. vs CL Pond. (MXN)"] = cuadro_sonr_fmt["Dif. vs CL Pond. (MXN)"].map("${:+,.2f}".format)
cuadro_sonr_fmt["Dif. vs CL Pond. (%)"] = cuadro_sonr_fmt["Dif. vs CL Pond. (%)"].map("{:+.2f}%".format)
display(cuadro_sonr_fmt)

# Gráfica comparativa de los métodos principales de SONR
metodos_grafica = cuadro_sonr.iloc[[0, 1, 2, 3, 5, 6, 7]].copy()
fig_sonr = px.bar(
    metodos_grafica,
    x="SONR Total Mercado (MXN)",
    y="Método",
    orientation="h",
    color="Método",
    text=metodos_grafica["SONR Total Mercado (MXN)"].map(lambda v: f"${v/1e6:,.1f} M"),
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title="Comparación de la Reserva SONR al 31/12/2025 por Metodología",
)
fig_sonr.update_traces(textposition="outside")
fig_sonr.update_layout(
    template="plotly",
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",
    font=dict(family="Arial", size=13, color="#2A4163"),
    showlegend=False,
    height=520,
    xaxis=dict(title="Monto de Reserva SONR (MXN)", tickformat=",.0f", range=[0, metodos_grafica["SONR Total Mercado (MXN)"].max() * 1.2]),
    yaxis=dict(title="", autorange="reversed"),
)
fig_sonr.show()

,Método,Fórmula / Criterio,Año 2023 (MXN),Año 2024 (MXN),Año 2025 (MXN),SONR Total Mercado (MXN),Dif. vs CL Pond. (MXN),Dif. vs CL Pond. (%)
0,1a. Chain-Ladder (Ponderado por Volumen),"Triángulo acumulado con factores f̂_j = ΣC_{i,...","$18,923,348.75","$204,971,260.23","$480,950,546.70","$704,845,155.68",$+0.00,+0.00%
1,1b. Chain-Ladder (Promedio Simple),Triángulo acumulado con factores f̄_j = media(...,"$23,176,945.62","$255,795,174.42","$646,000,694.39","$924,972,814.42","$+220,127,658.74",+31.23%
2,2a. Método de la Razón Ponderada / BF (S̄ 2015...,Σ Prima_i × S̄_2015-2022 × (1 - 1/F_i),"$27,549,294.90","$168,508,713.85","$743,779,499.31","$939,837,508.06","$+234,992,352.37",+33.34%
3,2b. Método de la Razón Ponderada / BF (S̄ 2015...,Σ Prima_i × S̄_2015-2025 × (1 - 1/F_i),"$25,895,296.73","$158,391,826.82","$699,124,638.45","$883,411,761.99","$+178,566,606.31",+25.33%
4,2c. Método de la Razón Directa — Ejercicio 202...,Prima_2025 × S̄_2015-2022 - Pagado_2025,$0.00,$0.00,"$1,895,721,800.80","$1,895,721,800.80","$+1,190,876,645.11",+168.96%
5,3a. Método Estatutario por Rezagos s/Prima (20...,Σ Prima_i × Σ_{j > 2025-i} r̄_j (tasas medias ...,"$26,635,967.59","$135,602,791.16","$553,198,732.83","$715,437,491.59","$+10,592,335.90",+1.50%
6,3b. Método Estatutario Ejercicio 2025 s/Prima ...,Prima_2025 × φ̄_prima (8.9930%),$0.00,$0.00,"$553,198,732.83","$553,198,732.83","$-151,646,422.85",-21.51%
7,3c. Método Estatutario Ejercicio 2025 s/Sinies...,Pagado_2025 × φ̄_sin (23.5526%),$0.00,$0.00,"$496,473,701.45","$496,473,701.45","$-208,371,454.23",-29.56%


Figure({
    'data': [{'hovertemplate': 'Método=%{y}<br>SONR Total Mercado (MXN)=%{x}<br>text=%{text}<extra></extra>',
              'legendgroup': '1a. Chain-Ladder (Ponderado por Volumen)',
              'marker': {'color': '#636EFA', 'pattern': {'shape': ''}},
              'name': '1a. Chain-Ladder (Ponderado por Volumen)',
              'orientation': 'h',
              'showlegend': True,
              'text': array(['$704.8 M'], dtype=object),
              'textposition': 'outside',
              'type': 'bar',
              'x': {'bdata': '9HrXsYoBxUE=', 'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['1a. Chain-Ladder (Ponderado por Volumen)'], dtype=object),
              'yaxis': 'y'},
             {'hovertemplate': 'Método=%{y}<br>SONR Total Mercado (MXN)=%{x}<br>text=%{text}<extra></extra>',
              'legendgroup': '1b. Chain-Ladder (Promedio Simple)',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 

---
# 7. Conclusiones Generales de las Reservas y del Análisis de la Base de Datos

Siguiendo el esquema analítico del proyecto (**observación $\rightarrow$ explicación plausible $\rightarrow$ implicación actuarial**), se sintetizan las conclusiones de los seis apartados:

### 1. Actualización monetaria (Apartado 1)
- **Observación:** Los montos históricos de emisión y siniestros entre 2015 y 2025 presentan una inflación acumulada de **$60.64\%$** desde diciembre de 2015 hasta diciembre de 2025 ($\text{factor INPC}_{2015} = 1.606368$).
- **Implicación actuarial:** Actualizar los flujos de primas según el año de emisión y cada pago de siniestro según su año efectivo de liquidación ($\text{año de origen} + \text{desarrollo}$) evita subestimar la severidad real de los años antiguos y garantiza que los factores de desarrollo del triángulo no estén distorsionados por inflación nominal.

### 2. Evolución de primas, comisiones y retención (Apartado 2)
- **Observación:** La prima emitida real creció de $\$2,479.72\text{ M}$ en 2015 a $\$6,151.44\text{ M}$ en 2025 ($+148.07\%$), impulsada a partir de 2023 por una fuerte concentración en **buques tanque petroleros** ($77\%$ de la emisión en 2025). Sin embargo, la **prima retenida** se mantuvo estable alrededor de $\$230.89\text{ M}$ ($3.75\%$ de retención en 2025) y las comisiones directas representaron apenas el $1.18\%$ en 2025 ($\$72.76\text{ M}$).
- **Implicación actuarial:** El mercado opera con una **cesión intensiva al reaseguro** en las grandes pólizas de cascos petroleros, lo que protege el capital retenido de las aseguradoras locales pero genera una alta exposición bruta y riesgo de contraparte reaseguradora.

### 3. Comportamiento de los siniestros y sus causas (Apartado 3)
- **Observación:** El año 2015 registró un máximo extraordinario de $\$4,285.57\text{ M}$ en siniestros ajustados (principalmente por fenómenos hidrometeorológicos / huracanes), mientras que en 2023 y 2024 los siniestros repuntaron por causas operativas (varaduras, encallamientos, avería particular y colisiones).
- **Implicación actuarial:** La cartera combina pérdidas frecuentes de origen operativo con eventos catastróficos de cola pesada, requiriendo coberturas de reaseguro catastrófico y reservas suficientes para años con alta severidad.

### 4. Factores de siniestralidad y segmentación por conglomerados (Apartado 4)
- **Observación:** La siniestralidad histórica del mercado promedia **$65.08\%$** en los años maduros (2015–2022) y **$61.18\%$** en el periodo completo (2015–2025), con un máximo en 2015 ($172.82\%$) y un mínimo en 2017 ($33.60\%$). La agrupación mediante el método de Ward identificó tres perfiles de embarcaciones según su nivel y volatilidad de siniestralidad, aunque con estabilidad temporal moderada entre subperiodos ($\text{ARI} = 0.278$).
- **Implicación actuarial:** La heterogeneidad entre tipos de embarcación justifica contrastar las reservas agregadas del mercado con estimaciones segmentadas por grupo de riesgo.

### 5. Reserva de Riesgos en Curso al 31 de diciembre de 2025 (Apartado 5)
- **Observación:** Sobre la prima de tarifa de 2025 ($\$6,078.68\text{ M}$), el **método exacto** arroja una RRC de **$\$5,244,839,068.74$** ($86.28\%$ de la prima de tarifa), el **método de los veinticuatroavos** estima **$\$5,284,812,484.50$** ($+0.76\%$ respecto al exacto), y el **método estatutario** ($\text{PND} \times \bar{S}$) sitúa la mejor estimación de siniestralidad futura entre **$\$3,208,653,187.09$** (con $\bar{S}_{2015\text{–}2025}$) y **$\$3,413,597,990.56$** (con $\bar{S}_{2015\text{–}2022}$).
- **Explicación e implicación actuarial:** La altísima proporción no devengada ($>86\%$) se explica porque el $79.99\%$ de la prima de 2025 se emitió en diciembre y permanece vigente durante casi todo 2026. El método exacto debe preferirse sobre veinticuatroavos al contar con fechas exactas de inicio y fin de vigencia, mientras que la brecha entre la PND ($\$5,244.84\text{ M}$) y el BEL estatutario ($\$3,413.60\text{ M}$) representa el margen técnico implícito antes de gastos de administración y margen de riesgo.

### 6. Reserva SONR al 31 de diciembre de 2025 (Apartado 6)
- **Observación:**
  - El método **Chain-Ladder ponderado por volumen** estima una SONR total de **$\$704,845,155.68$** (y **$\$924,972,814.42$** con promedio simple de factores), concentrada principalmente en el año de origen 2025 ($\$480.95\text{ M}$) y 2024 ($\$204.97\text{ M}$).
  - El **método estatutario por rezagos sobre prima emitida** arroja **$\$715,437,491.59$** ($+1.50\%$ respecto a Chain-Ladder ponderado), mostrando una notable consistencia con la proyección del triángulo acumulado.
  - El **método de la razón ponderado por desarrollo pendiente (Bornhuetter-Ferguson)** estima **$\$939,837,726.79$** con $\bar{S}_{2015\text{–}2022}$ ($+33.34\%$ vs. Chain-Ladder ponderado) y **$\$883,411,761.99$** con $\bar{S}_{2015\text{–}2025}$ ($+25.33\%$), debido al fuerte crecimiento de la prima emitida en 2025 ($\$6,151.44\text{ M}$) frente a una siniestralidad pagada en el año 0 de apenas $34.27\%$.
- **Implicación actuarial:** Cuando el volumen de primas crece abruptamente en el último año (como ocurrió en diciembre de 2025) y los pagos iniciales del año 0 aún son bajos porque la exposición se concentró al cierre del año, Chain-Ladder (que multiplica únicamente los pagos ya observados $C_{2025,0}$) puede subestimar los siniestros pendientes si existe rezago de reporte, mientras que los métodos basados en primas (**Razón / Bornhuetter-Ferguson** y **Estatutario sobre primas**) incorporan directamente el mayor volumen de exposición suscrita en 2025.